# Case file: 2700 Shattuck (359 units)

**For reporters.** The ZAB approved this mixed-use project in July 2026 after a debate over
waiving an affordable-housing requirement
([Daily Cal](https://www.dailycal.org/news/city/housing/berkeley-zoning-board-approves-shattuck-mixed-use-development-debates-waiving-affordable-housing-requirement/article_3c5c9878-a6f8-4cb6-adb0-53230ef59bb5.html)).
This notebook pulls the project's full history — and the citywide context for the two big
questions — from **berkeleybuild.com's public database**, computing every number in front of you.
Run each cell (Runtime → Run all). Edit any query; it's your copy.

*The record: an independent reconstruction from Berkeley's own permit data (every permit
2015–2026), audited row-by-row against the city's state filing (±130 units of ~4,200, every
difference named — [the audit](https://berkeleybuild.com/housing-audit.html)).*

In [ ]:
# Load the public database (fetched from berkeleybuild.com; ~4 MB)
import sqlite3, urllib.request
from pathlib import Path
DB = Path('berkeley_housing_public.db')
LOCAL = Path('docs/data/berkeley_housing_public.db')   # repo checkout fallback
if LOCAL.exists(): DB = LOCAL
elif not DB.exists():
    _req = urllib.request.Request('https://berkeleybuild.com/data/berkeley_housing_public.db',
                                  headers={'User-Agent': 'Mozilla/5.0'})   # the CDN 403s bare scripts
    DB.write_bytes(urllib.request.urlopen(_req, timeout=120).read())
con = sqlite3.connect(f'file:{DB}?mode=ro', uri=True)
import pandas as pd
print('projects on record:', con.execute('SELECT COUNT(*) FROM projects').fetchone()[0])

## 1 · The project's own clock
Every recorded event for 2700 Shattuck, oldest first.

In [ ]:
# auto-setup — safe to re-run; downloads only if needed (cells can run in ANY order)
import sqlite3, urllib.request; from pathlib import Path
import pandas as pd
if 'con' not in globals():
    _db = Path('docs/data/berkeley_housing_public.db')
    if not _db.exists():
        _db = Path('berkeley_housing_public.db')
        if not _db.exists():
            _req = urllib.request.Request('https://berkeleybuild.com/data/berkeley_housing_public.db',
                                          headers={'User-Agent': 'Mozilla/5.0'})   # the CDN 403s bare scripts
            _db.write_bytes(urllib.request.urlopen(_req, timeout=120).read())
    con = sqlite3.connect(f'file:{_db}?mode=ro', uri=True)
timeline = pd.read_sql("""
    SELECT substr(event_date,1,10) AS date, event_type_code AS event, record_key AS record,
           COALESCE(raw_description,'') AS detail
    FROM timeline WHERE project_key IN (SELECT project_key FROM projects WHERE address_display LIKE '2700 SHATTUCK%') ORDER BY 1""", con)
timeline

**How to read it:** SB 330 pre-application Nov 2023 → full application May/Jun 2024 → marked
*Incomplete* the same day (the intake ping-pong) → **deemed complete 2024-10-10** (the statutory
clock starts) → ZAB approval July 2026. That's ~**21 months in consideration** — the next cell
puts that on the citywide curve.

In [ ]:
# auto-setup — safe to re-run; downloads only if needed (cells can run in ANY order)
import sqlite3, urllib.request; from pathlib import Path
import pandas as pd
if 'con' not in globals():
    _db = Path('docs/data/berkeley_housing_public.db')
    if not _db.exists():
        _db = Path('berkeley_housing_public.db')
        if not _db.exists():
            _req = urllib.request.Request('https://berkeleybuild.com/data/berkeley_housing_public.db',
                                          headers={'User-Agent': 'Mozilla/5.0'})   # the CDN 403s bare scripts
            _db.write_bytes(urllib.request.urlopen(_req, timeout=120).read())
    con = sqlite3.connect(f'file:{_db}?mode=ro', uri=True)
clock = pd.read_sql("""
    SELECT address_display, COALESCE(NULLIF(units_built_or_permitted,0), proposed_units) AS total_units,
           CAST(julianday(entitled_date)-julianday(filed_date) AS INT) AS days_to_entitle
    FROM projects
    WHERE COALESCE(NULLIF(units_built_or_permitted,0), proposed_units)>=50 AND filed_date IS NOT NULL AND entitled_date IS NOT NULL
      AND julianday(entitled_date) > julianday(filed_date)""", con)
print(f"major projects with both dates: {len(clock)}")
print(f"median application->entitlement: {clock.days_to_entitle.median():.0f} days")
print(f"2700 Shattuck, application (2024-06) -> approval (2026-07): ~760 days and counting to a permit")
clock.sort_values('days_to_entitle', ascending=False).head(10)

## 2 · The affordability-waiver context
The debate was about the **State Density Bonus** (Gov. Code 65915): extra height/units in exchange
for below-market units, with waivers of local standards. Who else uses it in Berkeley:

In [ ]:
# auto-setup — safe to re-run; downloads only if needed (cells can run in ANY order)
import sqlite3, urllib.request; from pathlib import Path
import pandas as pd
if 'con' not in globals():
    _db = Path('docs/data/berkeley_housing_public.db')
    if not _db.exists():
        _db = Path('berkeley_housing_public.db')
        if not _db.exists():
            _req = urllib.request.Request('https://berkeleybuild.com/data/berkeley_housing_public.db',
                                          headers={'User-Agent': 'Mozilla/5.0'})   # the CDN 403s bare scripts
            _db.write_bytes(urllib.request.urlopen(_req, timeout=120).read())
    con = sqlite3.connect(f'file:{_db}?mode=ro', uri=True)
db = pd.read_sql("""
    SELECT address_display, COALESCE(NULLIF(units_built_or_permitted,0), proposed_units) AS total_units, status, substr(entitled_date,1,10) AS entitled
    FROM projects WHERE pathways LIKE '%density_bonus%' ORDER BY total_units DESC""", con)
print(f"density-bonus projects in the curated record: {len(db)} "
      "(text-mention floor across all planning filings: ~134 — see berkeleybuild.com)")
db.head(15)

## 3 · What approval means statistically — the waiting room
Approval is not a building. Entitled projects next need a **building permit**, and that queue is
where Berkeley's pipeline actually narrows:

In [ ]:
# auto-setup — safe to re-run; downloads only if needed (cells can run in ANY order)
import sqlite3, urllib.request; from pathlib import Path
import pandas as pd
if 'con' not in globals():
    _db = Path('docs/data/berkeley_housing_public.db')
    if not _db.exists():
        _db = Path('berkeley_housing_public.db')
        if not _db.exists():
            _req = urllib.request.Request('https://berkeleybuild.com/data/berkeley_housing_public.db',
                                          headers={'User-Agent': 'Mozilla/5.0'})   # the CDN 403s bare scripts
            _db.write_bytes(urllib.request.urlopen(_req, timeout=120).read())
    con = sqlite3.connect(f'file:{_db}?mode=ro', uri=True)
wr = pd.read_sql("""
    SELECT address_display, COALESCE(NULLIF(units_built_or_permitted,0), proposed_units) AS total_units, substr(entitled_date,1,10) AS entitled
    FROM projects
    WHERE entitled_date IS NOT NULL AND bp_issued_date IS NULL AND co_issued_date IS NULL
      AND status NOT IN ('withdrawn','denied','void') AND COALESCE(rhna_excluded,0)=0
    ORDER BY total_units DESC""", con)
print(f"THE WAITING ROOM today: {len(wr)} entitled projects, {wr.total_units.sum():,.0f} units, no permit")
print("Under BMC 23.404.060 an approval becomes lapse-ELIGIBLE one year after decision unless")
print("exercised — 2700 Shattuck's clock starts at its approval date.")
wr.head(12)

## 4 · The documents themselves — and how to read them with an AI
The record archives the project's **actual submissions** (plan sets, resubmittals) as permanent
PDFs — the city's own links expire; these don't. Download them below, then feed them to an AI
notebook: **NotebookLM** (upload → ask → cited answers) is the ten-minute path, or run an open
model **on your own laptop** so nothing leaves your machine —
[our guide](https://berkeleybuild.com/reporters-local-models.html). Either way: the model is a
reading assistant, not a source — verify anything you'll print against this database.

In [ ]:
# auto-setup — safe to re-run; downloads only if needed (cells can run in ANY order)
import sqlite3, urllib.request; from pathlib import Path
import pandas as pd
if 'con' not in globals():
    _db = Path('docs/data/berkeley_housing_public.db')
    if not _db.exists():
        _db = Path('berkeley_housing_public.db')
        if not _db.exists():
            _req = urllib.request.Request('https://berkeleybuild.com/data/berkeley_housing_public.db',
                                          headers={'User-Agent': 'Mozilla/5.0'})   # the CDN 403s bare scripts
            _db.write_bytes(urllib.request.urlopen(_req, timeout=120).read())
    con = sqlite3.connect(f'file:{_db}?mode=ro', uri=True)
docs = pd.read_sql("""
    SELECT title, substr(doc_date,1,10) AS date, page_count, store_url AS r2_url
    FROM documents WHERE project_key IN (SELECT project_key FROM projects WHERE address_display LIKE '2700 SHATTUCK%') ORDER BY doc_date""", con)
docs

In [ ]:
# Download every archived PDF for this project into ./pdfs/ (then drag them into NotebookLM,
# or point a local model at them — see the guide linked above)
from pathlib import Path
import urllib.request
Path('pdfs').mkdir(exist_ok=True)
for _, row in docs.dropna(subset=['r2_url']).iterrows():
    dest = Path('pdfs') / row.r2_url.split('/')[-1]
    if not dest.exists():
        req = urllib.request.Request(row.r2_url, headers={'User-Agent': 'Mozilla/5.0'})  # the archive 403s bare scripts
        dest.write_bytes(urllib.request.urlopen(req, timeout=120).read())
    print(f'{dest}  ({dest.stat().st_size/1e6:.1f} MB)')

## Cite & dig further
- Cite: *"berkeleybuild.com, an independent reconstruction of Berkeley's housing records from city
  permit data."*
- Browse this database with no code at all: [the reporters page](https://berkeleybuild.com/reporters.html)
  has one-click queries. The method, the audit, and the rebuild-it-yourself curriculum are all public.
- Something you need verified against the primary documents? Ask — that's what the project is for.